# Background subtraction

Two workflows are provided:

| Section           | Approach                                                                                                                    |
| ----------------- | --------------------------------------------------------------------------------------------------------------------------- |
| **Matched peaks** | Select matched peaks (known compositions) and subtract the intensity of each peak in a background sample from other samples |
| **All peaks**     | Bin peaks by m/z across all samples and subtract the intensity of each peak in a background sample from other samples       |


### Load peaks


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from mascope_sdk import MascopeClient


pio.templates.default = "plotly_dark"  # or "plotly_white"

mascope = MascopeClient(workspace="My Workspace")

# Load all peaks (adjust dataset/batches to your data)
peaks = mascope.load_peaks(dataset="My Dataset", batches="My Batch")

print(
    f"{len(peaks)} rows, {peaks['sample_item_name'].nunique()} samples, "
    f"{peaks['peak_id'].nunique()} unique peak_ids"
)
peaks

### Select background sample

Identify the sample whose signal will be subtracted from all other samples.
The cell below shows two methods: by exact sample name, and by keyword match. Uncomment the one that you want to use.


In [ ]:
# --- Select background sample ---
# Choose ONE of the methods below.

# Method 1: by exact sample name
background_name = "my-background-sample"

# Method 2: by keyword in the sample name (uncomment to use)
# background_name = peaks.loc[
#     peaks["sample_item_name"].str.contains("blank", case=False),
#     "sample_item_name",
# ].iloc[0]

# Resolve to sample_item_id
bg_mask = peaks["sample_item_name"] == background_name
if not bg_mask.any():
    available = sorted(peaks["sample_item_name"].unique())
    raise ValueError(f"Sample '{background_name}' not found.\nAvailable: {available}")
bg_id = peaks.loc[bg_mask, "sample_item_id"].iloc[0]

n_other = peaks["sample_item_name"].nunique() - 1
print(f"Background: '{background_name}' → {bg_id}")
print(f"Subtracting from {n_other} other samples")

---

## Section 1: Matched peaks

Subtract the background intensity of each matched **ion** from all other
samples. Only peaks with known compound/ion matches are used.

Heights are aggregated at the **ion level** (isotopes summed per ion) before
subtraction. Negative values after subtraction indicate the signal is below
the background level.


In [ ]:
# Filter to matched peaks only
matched = peaks[peaks["target_ion_formula"].notna()].copy()

if matched.empty:
    raise ValueError("No matched peaks found. Load data with compound matches.")

# Composite label: compound formula + ionization mechanism
matched["ion_formula_path"] = (
    matched["target_compound_formula"] + matched["ionization_mechanism"]
)

# Aggregate to ion level per sample (isotopes summed per ion)
ion_heights = (
    matched.groupby(
        [
            "sample_item_id",
            "sample_item_name",
            "datetime_utc",
            "target_ion_formula",
            "ion_formula_path",
        ]
    )["height"]
    .sum()
    .reset_index()
)

# Background intensities per ion
bg_ions = (
    ion_heights[ion_heights["sample_item_id"] == bg_id]
    .set_index("target_ion_formula")["height"]
    .rename("bg_height")
)

# Subtract background from all other samples
ion_sub = ion_heights[ion_heights["sample_item_id"] != bg_id].copy()
ion_sub["bg_height"] = ion_sub["target_ion_formula"].map(bg_ions).fillna(0)
ion_sub["height_subtracted"] = ion_sub["height"] - ion_sub["bg_height"]

# Summary per ion
summary = (
    ion_sub.groupby(["target_ion_formula", "ion_formula_path"])
    .agg(mean_height=("height", "mean"), bg_height=("bg_height", "first"))
    .assign(bg_fraction=lambda df: df["bg_height"] / df["mean_height"])
    .sort_values("bg_fraction", ascending=False)
)

n_bg = (summary["bg_height"] > 0).sum()
print(f"Background contributes to {n_bg} / {len(summary)} matched ions")
summary

In [ ]:
# Background fraction per ion
fig = px.bar(
    summary.reset_index().sort_values("bg_fraction"),
    x="bg_fraction",
    y="ion_formula_path",
    orientation="h",
    title="Background fraction per matched ion",
    hover_data=["target_ion_formula", "bg_height", "mean_height"],
)
fig.update_layout(
    xaxis_title="Background / Mean Sample Intensity",
    yaxis=dict(title="Ion"),
)
fig.show()

In [ ]:
# Raw vs background-subtracted timeseries
fig = px.scatter(
    ion_sub.sort_values("datetime_utc"),
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula", "bg_height"],
    title="Matched ions: raw intensity",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]", yaxis_type="log"
)
fig.show()

fig = px.scatter(
    ion_sub.sort_values("datetime_utc"),
    x="datetime_utc",
    y="height_subtracted",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula", "bg_height"],
    title="Matched ions: background subtracted",
)
fig.update_layout(
    xaxis_title="Datetime (UTC)",
    yaxis_title="Intensity (bg-subtracted) [cps]",
)
fig.show()

---

## Section 2: All peaks (m/z binning)

Subtract background at the **m/z bin** level for all peaks, regardless of
whether they have compound matches. Peaks are first grouped into m/z bins
using a gap-based algorithm, then the background sample's bin intensities
are subtracted from every other sample.


### m/z binning

Each sample identifies peaks independently, so the same physical signal has
slightly different m/z across samples. We cluster peaks into **m/z bins** using
a gap-based algorithm:

1. Sort all peaks by m/z
2. Walk through the sorted list; start a new bin whenever the gap to the next
   peak exceeds a ppm-based tolerance
3. Assign each peak its bin's **representative m/z** (intensity-weighted mean)


In [ ]:
# --- Parameters ---
mz_tolerance_ppm = 5  # max gap (in ppm) within a bin


def bin_peaks_by_mz(df: pd.DataFrame, tol_ppm: float) -> pd.DataFrame:
    """Assign an `mz_bin` label to each peak using gap-based clustering."""
    unique_peaks = df.drop_duplicates(subset=["sample_item_id", "peak_id"]).copy()

    sorted_df = unique_peaks.sort_values("mz").reset_index(drop=True)
    mz = sorted_df["mz"].values

    # Compute ppm gaps between consecutive peaks
    gaps_ppm = np.diff(mz) / mz[:-1] * 1e6

    # Assign bin ids: increment whenever gap exceeds tolerance
    bin_ids = np.zeros(len(mz), dtype=int)
    bin_ids[1:] = np.cumsum(gaps_ppm > tol_ppm)

    sorted_df["mz_bin"] = bin_ids
    return sorted_df


binned = bin_peaks_by_mz(peaks, mz_tolerance_ppm)

# Compute representative m/z per bin (intensity-weighted mean)
_wt = binned["mz"] * binned["height"]
bin_mz = (
    _wt.groupby(binned["mz_bin"]).sum()
    / binned["height"].groupby(binned["mz_bin"]).sum()
)
bin_mz.name = "mz_repr"

n_bins = binned["mz_bin"].nunique()
print(f"{n_bins} m/z bins from {len(binned)} peaks")

### Subtract background


In [ ]:
# Build intensity matrix: samples × m/z bins
intensity_matrix = binned.pivot_table(
    index="sample_item_id",
    columns="mz_bin",
    values="height",
    aggfunc="sum",
)

if bg_id not in intensity_matrix.index:
    raise ValueError(
        f"Background sample '{background_name}' has no peaks in the intensity matrix."
    )

# Extract background row (missing bins treated as 0)
bg_row = intensity_matrix.loc[bg_id].fillna(0)

# Subtract background from all other samples
subtracted_matrix = intensity_matrix.drop(bg_id).sub(bg_row)

n_bins_with_bg = (bg_row > 0).sum()
print(
    f"Intensity matrix: {subtracted_matrix.shape[0]} samples × "
    f"{subtracted_matrix.shape[1]} bins"
)
print(f"Background signal in {n_bins_with_bg} / {len(bg_row)} bins")
subtracted_matrix.head()

In [ ]:
# Mean mass spectrum: raw vs background vs subtracted
raw_mean = intensity_matrix.drop(bg_id).mean()

comparison = pd.DataFrame(
    {
        "mz": bin_mz[raw_mean.index].values,
        "raw": raw_mean.values,
        "background": bg_row[raw_mean.index].values,
        "subtracted": subtracted_matrix.mean().values,
    }
)

comp_long = comparison.melt(id_vars="mz", var_name="type", value_name="intensity")

fig = px.scatter(
    comp_long,
    x="mz",
    y="intensity",
    color="type",
    color_discrete_map={
        "raw": "#636EFA",
        "background": "#EF553B",
        "subtracted": "#00CC96",
    },
    title="Mean mass spectrum: raw, background, and background-subtracted",
)
fig.update_layout(
    xaxis_title="m/z [Th]",
    yaxis_title="Intensity (mean across samples) [cps]",
    yaxis_type="log",
)
fig.show()